# 01 — Likelihood Analysis of a bright point source with fermipy

This notebook follows the outline of <https://github.com/fermiPy/fermipy-extra/blob/master/notebooks/pg1553.ipynb> for the analysis of PG 1553+113. The original sample analysis is based on the LAT-team study of PG 1553+113 in [Abdo et al. 2010, ApJ 708, 1310](https://ui.adsabs.harvard.edu/abs/2010ApJ...708.1310A) and closely follows the FSSC [Likelihood Analysis with Python](https://fermi.gsfc.nasa.gov/ssc/data/analysis/scitools/python_tutorial.html) thread. fermipy docs: <https://fermipy.readthedocs.io>.

## Get the Data

For this thread the original data were extracted from the [LAT data server](https://fermi.gsfc.nasa.gov/cgi-bin/ssc/LAT/LATDataQuery.cgi) with the following selections (close to the parameters in Abdo+2010):

* Search Center (RA, Dec) = (238.929, 11.1901)
* Radius = 20°  
* Start Time (MET) = 239 557 417 s   (2008-08-04 15:43:37 UTC)
* Stop Time  (MET) = 256 970 880 s   (2009-02-22 04:48:00 UTC)
* Minimum Energy = 100 MeV
* Maximum Energy = 300 GeV

Our local FT1 / FT2 are already in `data/pg1553/` (you saw them in notebook 00). 

In [ ]:
import os
from pathlib import Path

# Working directory
ANADIR = Path(os.environ.get("LECTURE_DATA", "/opt/lecture-data")) / "pg1553"

os.environ['FERMI_DIFFUSE_DIR'] = os.path.join(
    os.environ['CONDA_PREFIX'],
    'share', 'fermitools', 'refdata', 'fermi', 'galdiffuse')

os.chdir(ANADIR)
print("working in:", os.getcwd())
print("\ncontents:")
for p in sorted(ANADIR.iterdir()):
    print(f"  {p.name}")

### Make a file list

fermipy needs a text file listing the FT1 chunks (and a similar one for FT2). We already wrote `ft1.txt` and `ft2.txt` into the working directory; let's verify them.

In [ ]:
print("=== ft1.txt ===")
print(open("ft1.txt").read())
print("=== ft2.txt ===")
print(open("ft2.txt").read())

## Make a config file

fermipy's analysis is configured through a [yaml](https://yaml.org) file. Ours is the same as the Fermi'ss analysis (cf. `fermipy-extras/data/pg1553/config.yaml`):

* `data.evfile` / `data.scfile`: the FT1 / FT2 listfiles.
* `binning`: 10° ROI side, 0.1°/pixel, 8 bins/decade in energy.
* `selection`: 100 MeV–300 GeV, MET window matches the LAT-server query, `evclass=128` (SOURCE), `evtype=3` (FRONT+BACK), `target='4FGL J1555.7+1111'`.
* `gtlike`: enable energy dispersion (`edisp=True`) with the `P8R3_SOURCE_V3` IRFs.
* `model`: 15° model-source ROI (slightly *larger* than the 10° fit ROI so PSF tails leak in correctly), galactic + isotropic diffuse from `$FERMI_DIFFUSE_DIR`, 4FGL catalog (fermipy 1.4.0 ships up to DR4 — the alias `'4FGL'` resolves to the latest available).

Full config reference: <https://fermipy.readthedocs.io/en/latest/config.html>.

In [ ]:
print(open("config.yaml").read())

## Start the analysis

We instantiate `GTAnalysis(config.yaml)`, then run `gta.setup()` which executes the rest of the FSSC analysis chain (`gtselect` → `gtmktime` → `gtltcube` → `gtbin` → `gtexpcube2` → `gtsrcmaps`) and stages every intermediate product under the working directory.

This is where the "magic" happens: fermipy loads the catalog, builds an XML source model for everything in the ROI, picks all the cuts and binnings, and runs the science tools for you. If files already exist on disk, it skips the corresponding step — so the *first* `gta.setup()` is slow (~15–30 min on a laptop for our time interval, dominated by `gtltcube` and `gtsrcmaps`); subsequent calls are seconds.

### Load up some modules

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

Silence a couple of harmless deprecation warnings emitted by the underlying tools.

In [ ]:
import warnings
try:
    from matplotlib import MatplotlibDeprecationWarning
    warnings.filterwarnings("ignore", category=MatplotlibDeprecationWarning)
except ImportError:
    pass
warnings.filterwarnings("ignore", category=FutureWarning)

### Import the GTAnalysis module from fermipy

Constructing the object reads the config and prints every parameter fermipy is going to use — including the many defaults you didn't override. Verbosity 3 = INFO (suppresses DEBUG).

In [ ]:
from fermipy.gtanalysis import GTAnalysis
gta = GTAnalysis("config.yaml", logging={"verbosity": 3})
matplotlib.interactive(True)

Let's take a look at the initial input event files,

In [ ]:
with open("ft1.txt") as f:
    input_files = [line.strip() for line in f if line.strip()]
print(input_files)

…and read one of them as an `astropy.table.Table` so we can browse the columns interactively.

In [ ]:
from astropy.table import Table
t = Table.read(input_files[0], hdu=1)
t

### The setup routine

`gta.setup()` runs every preprocessing step needed before likelihood. fermipy will *skip* any ancillary file that already exists — so re-running this cell after an interrupt is safe.

⏱ **First-run cost.** On our 6.5-month window expect ~15–30 min; the bottlenecks are `gtltcube` (livetime cube, all-sky × cos θ × time) and `gtsrcmaps` (one model map per source in the ROI, convolved with PSF and exposure). Re-runs that find the cached `*_00.fits` files complete in seconds.

In [ ]:
gta.setup(overwrite=True)

Before proceeding we have a quick look at what fermipy produced.

In [ ]:
import glob
for f in sorted(glob.glob("*.fits")):
    print(f)

Brief explanation of the contents of each file and its role in the analysis:

* **`ft1_00.fits`** — Event list. Generated by running `gtselect` and `gtmktime` on our input file list.
* **`bexpmap_00.fits`** — All-sky binned exposure map. Interpolated to create the exposure model when generating the source-map file.
* **`bexpmap_roi_00.fits`** — Binned exposure map *for the ROI*. Provided for visualisation (same binning as data and model maps).
* **`ccube_00.fits`** — Counts cube for the ROI (energy × lon × lat).
* **`ltcube_00.fits`** — Livetime cube. Map of the livetime over the whole sky as a function of incidence angle.
* **`srcmap_00.fits`** — Source-map cube. One map per ROI component, after convolution with exposure and PSF. (Energy dispersion is applied at run-time.)

The `_00` suffix is the analysis-component index — a multi-component analysis would have `_00`, `_01`, …, plus co-added maps without an index for visualisation.

To see one of these, open the counts cube and sum it over energy to make a 2-D sky image.

In [ ]:
from astropy.io import fits
from astropy.wcs import WCS

h = fits.open('ccube.fits')
h.info()

wcs = WCS(h[0].header).dropaxis(-1)   # drop the energy axis for plotting

counts = h[0].data
plt.subplot(projection=wcs)
im = plt.imshow(np.sum(counts, axis=0), interpolation='nearest',
                origin='lower', cmap='plasma')
plt.colorbar(im, label="counts")
plt.grid()
plt.gca().tick_params(direction='out')
plt.gca().set_xlabel("R.A. (deg)")
plt.gca().set_ylabel("Dec. (deg)")

Next, the sky map of the exposure for the ROI:

In [ ]:
exp = fits.open('bexpmap_roi_00.fits')
exp.info()
exposure = exp[0].data
wcs = WCS(exp[0].header).dropaxis(-1)

plt.subplot(projection=wcs)
im = plt.imshow(np.sum(exposure, axis=0), interpolation='nearest',
                origin='lower', cmap='plasma')
plt.colorbar(im, label=r"Exposure (cm$^{2}$ s)")
plt.grid()
plt.gca().tick_params(direction='out')
plt.gca().set_xlabel("R.A. (deg)")
plt.gca().set_ylabel("Dec. (deg)")

…and the energy dependence of the exposure for the central pixel.

In [ ]:
energy = exp[1].data
plt.loglog(energy, exposure[:, 50, 50])
plt.ylabel(r"Exposure (cm$^{2}$ s)")
plt.xlabel("Energy (MeV)")
plt.grid()

We can now inspect the state of the ROI prior to fitting with the `print_roi()` method.

In [ ]:
gta.print_roi()

Additional details about an individual source can be retrieved by printing the corresponding source object — here PG 1553+113.

In [ ]:
print(gta.roi['4FGL J1555.7+1111'])

## Do the likelihood fitting

Now that all of the ancillary files have been generated, we can move on to the actual fitting. The first thing you should do is free some of the sources, since they are all initially fixed. We free those near the centre of the ROI.

In [ ]:
gta.free_sources(free=False)                  # make sure everything is fixed first

# Free Normalization of all Sources within 3 deg of ROI center
gta.free_sources(distance=3.0, pars='norm')

# Free normalizations of isotropic and galactic diffuse components
gta.free_source('galdiff', pars='norm')
gta.free_source('isodiff')

In this simple analysis we leave the spectral *shapes* of nearby sources fixed but free the full spectral shape of our target.

In [ ]:
gta.free_source('4FGL J1555.7+1111')

Now actually run the fit. fermipy retries internally to coax the fitter into convergence.

In [ ]:
fit_results = gta.fit()

The dictionary returned by `fit()` carries diagnostics: the fit quality, the relative likelihood improvement, parameter correlations. Inspect the post-fit source object for PG 1553.

In [ ]:
print('Fit Quality:', fit_results['fit_quality'])
print(gta.roi['4FGL J1555.7+1111'])

And take another look at the ROI table and the full parameter list after the fit:

In [ ]:
gta.print_roi()

In [ ]:
gta.print_params()

You can save the state of the ROI to disk for later inspection. The first argument prefixes the output files; `make_plots=True` writes a stack of PNG diagnostics.

In [ ]:
gta.write_roi('fit0', make_plots=True)

We can also produce the *model map* (predicted counts per pixel under the best-fit model) and load it as a `gammapy.Map` (gammapy is a Cherenkov-telescope analysis package whose multi-dimensional Map class is convenient for this kind of cube; docs: <https://docs.gammapy.org>). We will talk about this in a moment.

In [ ]:
model_map = gta.write_model_map("fit0")

In [ ]:
print(model_map)

Plot the model map summed over the energy axis — predicted counts for *all* model components combined:

In [ ]:
axes = model_map[0].sum_over_axes(["energy"]).plot(stretch='log', add_cbar=True)
plt.show()

Plot each energy bin of the model map. This nicely illustrates the broad PSF at low energies (the source blob is degrees-wide at 100 MeV and shrinks to ≲ 0.1° at 100 GeV).

In [ ]:
axes = model_map[0].plot_grid(stretch='log', add_cbar=True,
                              figsize=(14, 18), ncols=4)
plt.show()

There are also several diagnostic plots `write_roi` saved to disk — counts map, x/y projections, counts spectrum, model map. Display them inline.

In [ ]:
from IPython.display import Image, display
pngs = sorted(glob.glob("fit0_*.png"))
for p in pngs:
    print(p)
    display(Image(p))

### Reading in the results

Since the results are pickled, you can reload them at any time. The on-disk numpy file mirrors the live `gta.roi` dictionary.

In [ ]:
c = np.load('fit0.npy', allow_pickle=True).flat[0]

The `sources` dictionary has an entry for each source in the model:

In [ ]:
sorted(c['sources'].keys())

And we can pull the flux, spectral parameters, and TS:

In [ ]:
print("flux        :", c['sources']['4FGL J1555.7+1111']['flux'])
print("param_names :", c['sources']['4FGL J1555.7+1111']['param_names'][:4])
print("param_values:", c['sources']['4FGL J1555.7+1111']['param_values'][:4])
print("TS          :", c['sources']['4FGL J1555.7+1111']['ts'])

The best-fit model SED is also stored in the `model_flux` sub-dictionary. Plot it.

In [ ]:
E       = np.array(c['sources']['4FGL J1555.7+1111']['model_flux']['energies'])
dnde    = np.array(c['sources']['4FGL J1555.7+1111']['model_flux']['dnde'])
dnde_hi = np.array(c['sources']['4FGL J1555.7+1111']['model_flux']['dnde_hi'])
dnde_lo = np.array(c['sources']['4FGL J1555.7+1111']['model_flux']['dnde_lo'])

plt.loglog(E, (E**2)*dnde,    'k--')
plt.loglog(E, (E**2)*dnde_hi, 'k')
plt.loglog(E, (E**2)*dnde_lo, 'k')
plt.xlabel('E [MeV]')
plt.ylabel(r'E$^2$ dN/dE [MeV cm$^{-2}$ s$^{-1}$]')
plt.show()

If you want SED *points* (binned likelihood per energy bin, with everything but the bin's normalisation frozen), there's `gta.sed`. Many options can be set in the config file or as keyword arguments — see [`fermipy.gtanalysis.GTAnalysis.sed`](https://fermipy.readthedocs.io/en/latest/fermipy.html#fermipy.gtanalysis.GTAnalysis.sed).

In [ ]:
gta.load_roi('fit0')                      
sed = gta.sed('4FGL J1555.7+1111')

You can save the state to a yaml file or just access it directly. This is also the way to get at the dictionary for any individual source.

In [ ]:
src = gta.roi['4FGL J1555.7+1111']

Plot the SED points on top of the best-fit model band.

In [ ]:
plt.loglog(E, (E**2)*dnde,    'k--')
plt.loglog(E, (E**2)*dnde_hi, 'k')
plt.loglog(E, (E**2)*dnde_lo, 'k')
plt.errorbar(np.array(sed['e_ctr']),
             sed['e2dnde'],
             yerr=sed['e2dnde_err'], fmt='o')
plt.xlabel('E [MeV]')
plt.ylabel(r'E$^{2}$ dN/dE [MeV cm$^{-2}$ s$^{-1}$]')
plt.show()

Looks like the last two points should be upper limits — they're noisy because PG 1553+113 is faint above ~30 GeV in 6.5 months. Replace them with proper 95 % upper limits (`e2dnde_ul95`, drawn as down-arrows).

In [ ]:
plt.loglog(E, (E**2)*dnde,    'k--')
plt.loglog(E, (E**2)*dnde_hi, 'k')
plt.loglog(E, (E**2)*dnde_lo, 'k')
plt.errorbar(sed['e_ctr'][:-2],
             sed['e2dnde'][:-2],
             yerr=sed['e2dnde_err'][:-2], fmt='o')
plt.errorbar(np.array(sed['e_ctr'][-2:]),
             sed['e2dnde_ul95'][-2:],
             yerr=0.2*sed['e2dnde_ul95'][-2:],
             fmt='o', uplims=True)
plt.xlabel('E [MeV]')
plt.ylabel(r'E$^{2}$ dN/dE [MeV cm$^{-2}$ s$^{-1}$]')
plt.show()

### Summary

There is much more functionality (TS maps, extension tests, event-type splitting, light-curves, …); see the fermipy docs and the rest of `fermipy-extras/notebooks/`. For PG 1553 specifically, the *Going further* options Meyer suggests:

- Re-run with `tmax: 'INDEF'` to use the full mission and reproduce the ~2.2-yr quasi-periodic flux modulation (Ackermann+ 2015, ApJL 813 L41) via `gta.lightcurve(...)` with monthly binning.
- Add an EBL absorption model on top of the intrinsic spectrum.

Next: the same machinery on a *less* clean and time-variable source — **TXS 0506+056**.